In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS olist.stg")

DataFrame[]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customers_df = spark.table("olist.ods.customers")

In [0]:
from pyspark.sql.functions import *

customers_stg = (
    spark.table("olist.ods.customers")
    .dropDuplicates()
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_zip_code_prefix"),
        trim(lower(col("customer_city"))).alias("customer_city"),
        trim(lower(col("customer_state"))).alias("customer_state")
    )
)

customers_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.customers")

In [0]:
geolocation_stg = (
    spark.table("olist.ods.geolocation")
    .dropDuplicates()
    .select(
        col("geolocation_zip_code_prefix"),
        col("geolocation_lat").cast("decimal(10,6)").alias("geolocation_lat"),
        col("geolocation_lng").cast("decimal(10,6)").alias("geolocation_lng"),
        trim(lower(col("geolocation_city"))).alias("geolocation_city"),
        trim(lower(col("geolocation_state"))).alias("geolocation_state")
    )
)

geolocation_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.geolocation")

In [0]:
orders_stg = (
    spark.table("olist.ods.orders")
    .dropDuplicates()
    .select(
        col("order_id"),
        col("customer_id"),

        coalesce(
            lower(trim(col("order_status"))),
            lit("unknown")
        ).alias("order_status"),

        when(col("order_approved_at").isNull(), 0).otherwise(1).alias("is_approved"),

        when(col("order_delivered_carrier_date").isNull(), 0).otherwise(1).alias("is_shipped"),

        when(col("order_delivered_customer_date").isNull(), 0).otherwise(1).alias("is_delivered"),

        col("order_purchase_timestamp").cast("timestamp").alias("order_purchase_timestamp"),

        col("order_approved_at").cast("timestamp").alias("order_approved_at"),

        col("order_delivered_carrier_date").cast("timestamp").alias("order_delivered_carrier_date"),

        col("order_delivered_customer_date").cast("timestamp").alias("order_delivered_customer_date"),

        col("order_estimated_delivery_date").cast("timestamp").alias("order_estimated_delivery_date")
    )
)

orders_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.orders")

In [0]:
order_items_stg = (
    spark.table("olist.ods.order_items")
    .dropDuplicates()
    .select(
        col("order_id"),
        col("order_item_id"),
        col("product_id"),
        col("seller_id"),

        col("shipping_limit_date").cast("timestamp").alias("shipping_limit_date"),

        col("price").cast("decimal(10,2)").alias("price"),

        col("freight_value").cast("decimal(10,2)").alias("freight_value")
    )
)

order_items_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.order_items")

In [0]:
order_payments_stg = (
    spark.table("olist.ods.order_payments")
    .dropDuplicates()
    .select(
        col("order_id"),
        col("payment_sequential"),

        trim(lower(col("payment_type"))).alias("payment_type"),

        col("payment_installments").cast("integer").alias("payment_installments"),

        col("payment_value").cast("decimal(10,2)").alias("payment_value")
    )
)

order_payments_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.order_payments")

In [0]:
order_reviews_stg = (
    spark.table("olist.ods.order_reviews")
    .dropDuplicates()
    .select(
        coalesce(col("review_id"), lit("unknown")).alias("review_id"),

        coalesce(col("order_id"), lit("unknown")).alias("order_id"),

        coalesce(
            col("review_score").cast("integer"),
            lit(0)
        ).alias("review_score"),

        when(col("review_comment_title").isNull(), 0).otherwise(1).alias("has_review_title"),

        when(col("review_comment_message").isNull(), 0).otherwise(1).alias("has_review_message"),

        coalesce(
            trim(lower(col("review_comment_title"))),
            lit("unknown")
        ).alias("review_comment_title"),

        coalesce(
            trim(lower(col("review_comment_message"))),
            lit("unknown")
        ).alias("review_comment_message"),

        col("review_creation_date").cast("timestamp").alias("review_creation_date"),

        col("review_answer_timestamp").cast("timestamp").alias("review_answer_timestamp")
    )
)

order_reviews_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.order_reviews")

In [0]:
products_stg = (
    spark.table("olist.ods.products")
    .dropDuplicates()
    .select(
        coalesce(col("product_id"), lit("unknown")).alias("product_id"),

        coalesce(
            trim(lower(col("product_category_name"))),
            lit("unknown")
        ).alias("product_category_name"),

        coalesce(col("product_name_lenght").cast("integer"), lit(0)).alias("product_name_length"),

        coalesce(col("product_description_lenght").cast("integer"), lit(0)).alias("product_description_length"),

        coalesce(col("product_photos_qty").cast("integer"), lit(0)).alias("product_photos_qty"),

        coalesce(col("product_weight_g").cast("double"), lit(0)).alias("product_weight_g"),

        coalesce(col("product_length_cm").cast("double"), lit(0)).alias("product_length_cm"),

        coalesce(col("product_height_cm").cast("double"), lit(0)).alias("product_height_cm"),

        coalesce(col("product_width_cm").cast("double"), lit(0)).alias("product_width_cm")
    )
)

products_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.products")

In [0]:
sellers_stg = (
    spark.table("olist.ods.sellers")
    .dropDuplicates()
    .select(
        col("seller_id"),
        col("seller_zip_code_prefix"),

        trim(lower(col("seller_city"))).alias("seller_city"),

        trim(lower(col("seller_state"))).alias("seller_state")
    )
)

sellers_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.sellers")

In [0]:
category_translation_stg = (
    spark.table("olist.ods.category_translation")
    .dropDuplicates()
    .select(
        trim(lower(col("product_category_name"))).alias("product_category_name"),
        trim(lower(col("product_category_name_english"))).alias("product_category_name_english")
    )
)

category_translation_stg.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.stg.category_translation")